# Data Preprocessing — UN International Migrant Stock 2024
**Dataset:** `undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx`  
**Authors:** Noman Shahzad · Visiliki · Stephan  
**Steps covered:**
1. Load the raw Excel file
2. Understand the raw structure
3. Extract and rename headers
4. Melt wide format → long (tidy) format
5. Add sex labels
6. Clean & filter data
7. Handle missing values
8. Add flag columns
9. Save output as `table1.parquet`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

FILE = Path("undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx")
print("File exists:", FILE.exists())
print("File size:", FILE.stat().st_size / 1e6, "MB")


## Step 1 — Load Raw Excel & Inspect Structure
The Excel file has 11 sheets. **Table 1** is the main data sheet containing bilateral migration flows (origin → destination) for every census year from 1990 to 2024, broken down by sex (both sexes, males, females).

The raw sheet has:
- **Rows 1–7**: title, citation and copyright metadata — skipped
- **Row 8**: sex group label for columns 8–15 (`both sexes`)
- **Row 10**: sex group labels for columns 16–23 (`males`) and 24–31 (`females`)
- **Row 11**: actual column headers (Index, destination, origin, year columns)
- **Rows 12+**: data rows


In [ ]:
# Load Table 1 skipping the first 10 rows of metadata
# Row 11 (index 10) is the real header
raw = pd.read_excel(
    FILE,
    sheet_name="Table 1",
    header=10,          # 0-indexed: row 11
    engine="openpyxl",
)

print("Raw shape:", raw.shape)
print()
print("Columns (first 16):")
for i, c in enumerate(raw.columns[:16]):
    print(f"  [{i:02d}] {c!r}")


## Step 2 — Understand Column Layout
The 31 columns are arranged as:

| Cols | Content |
|------|---------|
| 0–6  | Metadata: Index, Destination name, Coverage, Data type, Destination code, Origin name, Origin code |
| 7–14 | **Both sexes**: 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024 |
| 15–22| **Males**: 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024 |
| 23–30| **Females**: 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024 |

We need to split this into three separate blocks and label each with the correct sex, then stack them vertically (melt to long format).


In [ ]:
YEARS = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]
META  = ["row_id", "destination", "coverage", "data_type",
         "destination_code", "origin", "origin_code"]

# Rename all 31 columns clearly
new_cols = (
    META +
    [f"both_{y}" for y in YEARS] +
    [f"male_{y}" for y in YEARS] +
    [f"female_{y}" for y in YEARS]
)

raw.columns = new_cols
print("Renamed columns:")
print(raw.columns.tolist())
print()
print("Shape:", raw.shape)


## Step 3 — Remove Metadata Rows
The first few rows of the data contain leftover header/label text from the Excel sheet. We keep only numeric `row_id` rows (actual data records).


In [ ]:
# Drop rows where row_id is not a number (leftover header text rows)
raw["row_id"] = pd.to_numeric(raw["row_id"], errors="coerce")
raw = raw.dropna(subset=["row_id"]).copy()
raw["row_id"] = raw["row_id"].astype(int)

# Also drop rows where destination or origin is blank
raw = raw.dropna(subset=["destination", "origin"])

print("Shape after removing metadata rows:", raw.shape)
print()
print("Sample rows:")
raw[META].head(5)


## Step 4 — Melt Wide → Long (Tidy) Format
Currently each row has 24 year columns (8 years × 3 sexes). We reshape to **tidy format** where each row represents one observation: one origin–destination pair, for one year, for one sex group. This is essential for Plotly/Dash filtering and aggregation.

**Before melt:** 1 row = 1 origin–destination pair with 24 values  
**After melt:** 1 row = 1 origin–destination–year–sex combination


In [ ]:
frames = []

for sex_label, prefix in [("both_sexes", "both"), ("males", "male"), ("females", "female")]:
    # Select meta columns + the 8 year columns for this sex
    year_cols = [f"{prefix}_{y}" for y in YEARS]
    subset = raw[META + year_cols].copy()

    # Rename year columns to plain year integers for melting
    subset = subset.rename(columns={f"{prefix}_{y}": y for y in YEARS})

    # Melt: each year becomes its own row
    melted = subset.melt(
        id_vars=META,
        value_vars=YEARS,
        var_name="year",
        value_name="migrant_stock",
    )
    melted["sex"] = sex_label
    frames.append(melted)

# Stack all three sex groups
df = pd.concat(frames, ignore_index=True)

print("Long-format shape:", df.shape)
print()
print("Columns:", df.columns.tolist())
print()
print("Sex values:", df["sex"].unique())
print("Year values:", sorted(df["year"].unique()))


## Step 5 — Convert Data Types
- `year` → integer  
- `migrant_stock` → numeric (some cells contain `..` or text markers meaning missing data)  
- `destination_code` / `origin_code` → integer  


In [ ]:
# Year to int
df["year"] = df["year"].astype(int)

# migrant_stock: coerce non-numeric (e.g. '..') to NaN
df["migrant_stock"] = pd.to_numeric(df["migrant_stock"], errors="coerce")

# Codes to int where possible
df["destination_code"] = pd.to_numeric(df["destination_code"], errors="coerce")
df["origin_code"]      = pd.to_numeric(df["origin_code"], errors="coerce")

print("Data types:")
print(df.dtypes)
print()
print("Migrant stock range:")
print(df["migrant_stock"].describe())


## Step 6 — Handle Missing Values
The UN dataset uses structured missingness — not all origin–destination pairs are observed in every year. We document the extent of missing data rather than imputing, since fabricating migration counts would be analytically misleading.


In [ ]:
total      = len(df)
missing    = df["migrant_stock"].isna().sum()
pct        = missing / total * 100

print(f"Total rows:          {total:,}")
print(f"Missing stock values: {missing:,}  ({pct:.1f}%)")
print()

# Missing by year
print("Missing by year:")
print(
    df.groupby("year")["migrant_stock"]
    .apply(lambda x: x.isna().sum())
    .rename("missing_count")
)
print()

# Missing by sex
print("Missing by sex:")
print(
    df.groupby("sex")["migrant_stock"]
    .apply(lambda x: x.isna().sum())
    .rename("missing_count")
)


## Step 7 — Add Quality Flag Columns
The original dataset includes flag columns in the metadata indicating special data conditions:
- **Flag B** (`flag_B_foreign_born`) — stock measured as foreign-born population
- **Flag C** (`flag_C_foreign_citizens`) — stock measured as foreign citizens  
- **Flag I** (`flag_I_imputed`) — value is imputed/estimated  
- **Flag R** (`flag_R_refugee_adjustment`) — includes refugee adjustment

We add these as boolean columns for transparency in the dashboard.


In [ ]:
# Add placeholder flag columns (set False by default)
# In the real dataset these come from the 'Coverage' and 'Data type' columns
df["flag_B_foreign_born"]        = df["coverage"].str.contains("B", na=False)
df["flag_C_foreign_citizens"]    = df["coverage"].str.contains("C", na=False)
df["flag_I_imputed"]             = df["data_type"].str.contains("I", na=False)
df["flag_R_refugee_adjustment"]  = df["data_type"].str.contains("R", na=False)

print("Flag value counts:")
for col in ["flag_B_foreign_born","flag_C_foreign_citizens",
            "flag_I_imputed","flag_R_refugee_adjustment"]:
    print(f"  {col}: {df[col].sum():,} rows flagged")


## Step 8 — Filter to Country-Level Rows
The UN dataset includes rows for:
- Individual **countries** (location codes < 900)
- **Regions** and aggregates (codes ≥ 900: World, Sub-Saharan Africa, etc.)

For the dashboard we need both, but we add a helper column `is_country` so we can filter easily in the app. The dashboard uses `destination_code < 900` and `origin_code < 900` to show only country-to-country flows.


In [ ]:
# Add country flag
df["is_country_destination"] = df["destination_code"] < 900
df["is_country_origin"]      = df["origin_code"]      < 900

# Country-only subset stats
country_df = df[df["is_country_destination"] & df["is_country_origin"]]

print(f"Full dataset rows:            {len(df):,}")
print(f"Country-only rows:            {len(country_df):,}")
print(f"Unique destination countries: {country_df['destination'].nunique()}")
print(f"Unique origin countries:      {country_df['origin'].nunique()}")
print(f"Years covered:                {sorted(country_df['year'].unique())}")


## Step 9 — Final Column Order & Summary
Arrange columns in a logical order before saving.


In [ ]:
FINAL_COLS = [
    "row_id",
    "destination",
    "coverage",
    "data_type",
    "destination_code",
    "origin",
    "origin_code",
    "sex",
    "year",
    "migrant_stock",
    "is_country_destination",
    "is_country_origin",
    "flag_B_foreign_born",
    "flag_C_foreign_citizens",
    "flag_I_imputed",
    "flag_R_refugee_adjustment",
]

df = df[FINAL_COLS].copy()

print("Final dataset shape:", df.shape)
print()
print("Final columns:")
for c in df.columns:
    print(f"  {c}")
print()
print("Sample rows (both_sexes, country-level, 2024):")
df[
    (df["sex"] == "both_sexes") &
    (df["is_country_destination"]) &
    (df["is_country_origin"]) &
    (df["year"] == 2024)
].head(5)[["destination","origin","year","sex","migrant_stock"]]


## Step 10 — Save as Parquet
We save the cleaned tidy dataset as **Parquet** format. Parquet is:
- **Fast** to read (columnar storage, compressed)
- **Typed** (preserves int/float/string dtypes)
- **Small** (much smaller than CSV for this dataset)

The dashboard (`app.py`) loads `table1.parquet` directly.


In [ ]:
df.to_parquet("table1.parquet", engine="fastparquet", index=False)

import os
size_mb = os.path.getsize("table1.parquet") / 1e6
print(f"✅ Saved: table1.parquet  ({size_mb:.2f} MB)")
print(f"   Rows: {len(df):,}")
print(f"   Cols: {len(df.columns)}")


## Step 11 — Verification
Quick sanity check: reload the parquet and confirm the data looks correct.


In [ ]:
verify = pd.read_parquet("table1.parquet", engine="fastparquet")

print("Reloaded shape:", verify.shape)
print()
print("Top 5 destinations in 2024 (both sexes, country-level):")
(
    verify[
        (verify["sex"] == "both_sexes") &
        (verify["destination_code"] < 900) &
        (verify["origin_code"] < 900) &
        (verify["year"] == 2024)
    ]
    .groupby("destination")["migrant_stock"]
    .sum()
    .nlargest(5)
    .reset_index()
    .rename(columns={"migrant_stock": "total_migrants"})
    .assign(total_millions=lambda x: (x["total_migrants"]/1e6).round(1))
)


## Preprocessing Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Load Excel Table 1, skip 10 metadata rows | Raw 28,041 × 31 DataFrame |
| 2 | Rename 31 columns (7 meta + 8 both + 8 male + 8 female) | Clear column names |
| 3 | Remove non-data rows (leftover headers) | Clean row_id column |
| 4 | Melt wide → long (3 sex groups × 8 years) | Tidy format |
| 5 | Convert types (year→int, stock→float, codes→int) | Correct dtypes |
| 6 | Document missing values (UN uses structured missingness) | No imputation |
| 7 | Add quality flag columns (B, C, I, R) from coverage/data_type | Transparency |
| 8 | Add `is_country_destination` / `is_country_origin` flags | Easy filtering |
| 9 | Arrange final column order | Clean schema |
| 10 | Save as `table1.parquet` | Fast load for dashboard |
| 11 | Verify reload | ✅ Confirmed correct |

**Output:** `table1.parquet` — ready to use with `app.py` (the Dash dashboard).
